# 🎬 Supervised Learning (Binary Classification)

This notebook demonstrates a supervised learning demo using the **Kaggle Heart Disease** dataset built with the **scikit-learn** library.

**Goal:** Predict whether a patient has heart disease (`target = 1`) or not (`target = 0`) based on clinical features.

In [ ]:
# ====================
# STEP 1: Import libs
# ====================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score
)
import numpy as np

# ===========================================
# STEP 2: Read csv file
# ===========================================
csv_path = "./heart.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print(df.head())

# ===========================================
# STEP 3: Assess class balance
# ===========================================
print("\nClass distribution (target column):")
print(df['target'].value_counts())

print("\nClass proportions:")
print(df['target'].value_counts(normalize=True).round(3))


In [ ]:
# ===========================================
# STEP 4: Data Preprocessing / Data Cleaning
# ===========================================
if 'id' in df.columns:
    df = df.drop('id', axis=1)

X = df.drop('target', axis=1)
y = df['target']

# ====================
# STEP 5: Split data
# =====================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ======================
# STEP 6: Scale features
# ======================
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# ======================
# STEP 7: Train model
# ======================
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train_s, y_train)

print("Model training complete!")

## Evaluate

In [ ]:
# ======================
# STEP 8: Evaluate
# ======================
y_pred = clf.predict(X_test_s)
y_proba = clf.predict_proba(X_test_s)[:,1]

print("\nConfusion Matrix (TN, FP / FN, TP):")
print("***********************\n")
cm = confusion_matrix(y_test, y_pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"\n  TP={tp}  TN={tn}  FP={fp}  FN={fn}")

print("\nAverage Accuracy:")
print("***********************\n")
print(round(accuracy_score(y_test, y_pred), 4))

print("\nBalanced Accuracy:")
print("***********************\n")
print(round(balanced_accuracy_score(y_test, y_pred), 4))

print("\nClassification Report (Precision, Recall, F1-score):")
print("***********************\n")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Heart Disease']))

print("\nROC AUC:")
print("***********************\n")
print(round(roc_auc_score(y_test, y_proba), 4))

## Three sample input/output *examples*



In [ ]:
# ===========================================
# STEP 9: Three sample input/output examples
# =============================================
print("Goal: Predict absence (0) or presence (1) of heart disease")
print("*****************************************************************\n\n")

np.random.seed(42)
sample_idx = np.random.choice(range(X_test.shape[0]), size=3, replace=False)
sample_features = X_test.iloc[sample_idx]
sample_true = y_test.iloc[sample_idx].values
sample_pred = clf.predict(X_test_s[sample_idx])
sample_prob = clf.predict_proba(X_test_s[sample_idx])[:, 1]

display_cols = ['age', 'sex', 'cp', 'trestbps', 'chol']
example_df = sample_features[display_cols].copy()
example_df['True_Label'] = sample_true
example_df['Predicted_Label'] = sample_pred
example_df['Prob_Disease'] = sample_prob.round(3)

print(example_df)
print("\n\nInterpretation:")
print("***********************\n")
print(" - True_Label: 0 = Absence of heart disease, 1 = Presence of heart disease")
print(" - Predicted_Label: model's classification")
print(" - Prob_Disease: model's confidence in the patient having heart disease (class 1)")